# Week 01 — Small Python Data-Processing App

This notebook loads a raw CSV (also works with a list), cleans it with named functions, and processes numeric columns with **NumPy**.

**Colab:** Runtime → Run all. If `src` is not on the path, the setup cell locates `week1/` next to this notebook.

In [1]:
import sys
from pathlib import Path

try:
    import numpy as np
except ImportError:
    import subprocess
    subprocess.check_call([sys.executable, "-m", "pip", "install", "numpy"])
    import numpy as np

here = Path.cwd().resolve()
candidates = [here, here.parent, here.parent.parent]
week01 = next((p for p in candidates if (p / "src").is_dir() and (p / "data").is_dir()), None)
if week01 is None:
    raise FileNotFoundError("Could not find week1 (needs src/ and data/).")
sys.path.insert(0, str(week01))
print("Project folder:", week01)
print("numpy", np.__version__, "ready")

Project folder: /week1
numpy 2.x ready


## 1. Load a raw dataset (CSV or list)

The CSV is stored in the repo at `data/raw_students.csv`. `load_from_list` shows the same pipeline with data defined in code.

In [2]:
from src.load import load_csv, load_from_list

raw_rows = load_csv()
list_rows = load_from_list()

print("CSV rows:", len(raw_rows))
print("First raw row:", raw_rows[0])
print("List rows:", len(list_rows))

CSV rows: 15
First raw row: {'name': '  Ada Lovelace', 'age': '36', 'city': 'london', 'score': '88', 'hours_studied': '12'}
List rows: 3


## 2. Clean with named functions

Cleaning covers missing values, duplicates, text/case fixes, type conversion, and outliers.

In [3]:
from src.clean import (
    strip_text_fields,
    normalize_missing_values,
    fix_text_case,
    convert_column_types,
    drop_duplicate_rows,
    remove_outliers,
    drop_incomplete_records,
    clean_dataset,
)

step = strip_text_fields(raw_rows)
step = normalize_missing_values(step)
step = fix_text_case(step)
step = convert_column_types(step)
print("After strip + missing + case + types:", len(step))

step = drop_duplicate_rows(step)
print("After dropping duplicates:", len(step))

step = remove_outliers(step)
print("After removing outliers:", len(step))

cleaned = drop_incomplete_records(step)
print("After dropping incomplete rows:", len(cleaned))
print("Cleaned names:", [row["name"] for row in cleaned])

assert cleaned == clean_dataset(raw_rows)

After strip + missing + case + types: 15
After dropping duplicates: 14
After removing outliers: 10
After dropping incomplete rows: 7
Cleaned names: ['Ada Lovelace', 'Alan Turing', 'Grace Hopper', 'Guido Van Rossum', 'Bjarne Stroustrup', 'Bob', 'Eve']


## 3. Process with NumPy

`mean`, `median`, `std`, `percentile`, and `np.where` are each wrapped in a named function.

In [4]:
from src.process import (
    scores_array,
    mean_score,
    median_score,
    std_score,
    score_percentile,
    high_scores,
    summarize,
)

scores = scores_array(cleaned)
print("scores:", scores)
print(summarize(cleaned))
print("high scores (np.where):", high_scores(scores))
print("mean / median / std / p75:", mean_score(scores), median_score(scores), std_score(scores), score_percentile(scores))

scores: [88. 91. 95. 79. 86. 73. 90.]
{'count': 7, 'mean_score': 86.0, 'median_score': 88.0, 'std_score': 7.010196655077313, 'p75_score': 90.5, 'high_score_count': 5, 'mean_hours': 10.571428571428571}
high scores (np.where): [88. 91. 95. 86. 90.]


In [5]:
from src.clean import clean_dataset as clean_again

from_list = clean_again(list_rows)
print("List path cleaned rows:", len(from_list))
print("Done.")

List path cleaned rows: 3
Done.
